In [67]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import torch.optim as optimizer

import seaborn as sns

In [68]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

In [69]:
#This will state whether neural network is set up to use GPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [70]:
#Sampling the data so test runs don't take so long
dataSample = df.sample(frac=0.4)
#define X (data features) and y (target)
#X is all useful features
#y is target (final column 'default payment next month')

#Default Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade
features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

#Other features to drop:
# dropColumns = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id'] #Too specific categorical, would produce thousands of columns
#I decided we keep geo_level_1_id as there are only 30 and it is overal region, may have more value
#but we need to make it categorical to generate one-hot encodings
# features['geo_level_1_id'] = features['geo_level_1_id'].astype('category')

# features = features.drop(dropColumns, axis = 1)
#Data adjustment
# encoder = OneHotEncoder(sparse_output=False, drop='first')

#These tables are in string type and must be converted
# categoryTables = ['geo_level_1_id','land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
categoryTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
one_hot_tables = pd.get_dummies(features[categoryTables], dtype=int, drop_first=True)




#Now we replace our string tables with the encoded tables
X = features.drop(categoryTables, axis = 1)
X = X.join(one_hot_tables)
print(X.info())

# scaler = MinMaxScaler()
scaler = RobustScaler()
X = scaler.fit_transform(X)


y = dataSample['damage_grade']

<class 'pandas.core.frame.DataFrame'>
Index: 104240 entries, 31149 to 32772
Data columns (total 60 columns):
 #   Column                                  Non-Null Count   Dtype
---  ------                                  --------------   -----
 0   geo_level_1_id                          104240 non-null  int64
 1   geo_level_2_id                          104240 non-null  int64
 2   geo_level_3_id                          104240 non-null  int64
 3   count_floors_pre_eq                     104240 non-null  int64
 4   age                                     104240 non-null  int64
 5   area_percentage                         104240 non-null  int64
 6   height_percentage                       104240 non-null  int64
 7   has_superstructure_adobe_mud            104240 non-null  int64
 8   has_superstructure_mud_mortar_stone     104240 non-null  int64
 9   has_superstructure_stone_flag           104240 non-null  int64
 10  has_superstructure_cement_mortar_stone  104240 non-null  int64
 11  ha

In [71]:
#splitting data and 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [72]:


X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)

X_test_tensor = torch.tensor(X_test)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [73]:
print(X_train_tensor)
print(X_train_tensor.size(1))

tensor([[ 1.0000,  0.2782, -0.0388,  ...,  0.0000,  0.0000,  0.0000],
        [-0.1429, -0.0157,  0.5008,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.3571, -0.6676,  0.5827,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-0.3571,  0.3752, -0.4940,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.6429, -0.1284, -0.5114,  ...,  0.0000,  0.0000,  0.0000],
        [-0.4286, -0.1555, -0.0690,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.float64)
60


In [76]:
#data loader loads data (shocking!)
train_loader = DataLoader(dataset=X_train_tensor, batch_size=40)

In [83]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #define size of out input (feature number)
        inputSize = X_train_tensor.size(1)
        
        self.flatten = nn.Flatten()
        
        self.linear_relu_stack = nn.Sequential(#The sequential structure of the earthquake classifier
            
            nn.Linear(inputSize, 30), #Bias is on by default if not explicitly set off
            nn.ReLU(),
            nn.Linear(30, 9),
            nn.ReLU(),
            nn.Linear(9, 3),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
    # def classifier(self,z):
    #     return nnf.softmax(self(z))

    def parameters(self):
        return self.linear_relu_stack

In [84]:
model = NeuralNetwork()

In [81]:
criterion = nn.CrossEntropyLoss() #this handles softmax for us so we don't need it in our layers

In [87]:
adamOptimizer = optimizer.Adam(model.parameters(), lr=0.001)

TypeError: optimizer can only optimize Tensors, but one of the params is torch.nn.modules.linear.Linear

In [77]:
# Train the model
num_epochs = 150
history = {'loss': [], 'val_loss': []}
#training
for epoch in range(num_epochs):
    model.train()
    adamOptimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)  
    loss.backward()
    adamOptimizer.step()
    history['loss'].append(loss.item())
    #testing
    model.eval()
    with torch.no_grad():
        outputs_val = model(X_test_tensor)
        val_loss = criterion(outputs_val, y_test_tensor)  
        history['val_loss'].append(val_loss.item())

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Validation Loss: {val_loss.item():.4f}')

NameError: name 'adamOptimizer' is not defined